# 翻前策略加载验证

验证 `bayes_poker.strategy.preflop` 模块能正确加载真实策略数据。

In [ ]:
import logging
from pathlib import Path

from bayes_poker.strategy.preflop import (
    parse_all_strategies,
    PreflopStrategy,
)

# 设置日志级别以查看解析进度
logging.basicConfig(level=logging.INFO)

STRATEGY_ROOT = Path("/home/autumn/project/gg_handhistory/preflop_strategy")

## 1. 加载所有策略

In [ ]:
strategies = parse_all_strategies(STRATEGY_ROOT)
print(f"共加载 {len(strategies)} 个策略")

## 2. 策略概览

In [ ]:
for strategy in strategies:
    print(f"\n策略: {strategy.name}")
    print(f"  来源目录: {strategy.source_dir}")
    print(f"  Stack sizes: {strategy.stack_sizes()}")
    print(f"  总节点数: {strategy.node_count()}")
    for stack_bb in strategy.stack_sizes():
        print(f"    - {stack_bb} BB: {strategy.node_count(stack_bb)} 个节点")

## 3. 查看根节点（开局决策点）

In [ ]:
# 选取第一个策略
if strategies:
    strategy = strategies[0]
    stack_bb = strategy.stack_sizes()[0] if strategy.stack_sizes() else None
    
    if stack_bb:
        root_node = strategy.get_node(stack_bb, "")
        if root_node:
            print(f"策略: {strategy.name}, Stack: {stack_bb} BB")
            print(f"根节点行动位置: {root_node.acting_position}")
            print(f"可用行动数: {len(root_node.actions)}")
            print("\n各行动详情:")
            for action in root_node.actions:
                print(f"  - {action.action_code} ({action.action_type})")
                print(f"      频率: {action.total_frequency:.1%}")
                print(f"      EV: {action.total_ev:.3f}")
                print(f"      Combos: {action.total_combos:.1f}")

## 4. 探索行动历史节点

In [ ]:
# 列出某个 stack 下的前 10 个节点
if strategies:
    strategy = strategies[0]
    stack_bb = strategy.stack_sizes()[0] if strategy.stack_sizes() else None
    
    if stack_bb:
        nodes = strategy.nodes_by_stack.get(stack_bb, {})
        print(f"策略 {strategy.name}, Stack {stack_bb} BB 的节点 (前10个):")
        for i, (history, node) in enumerate(list(nodes.items())[:10]):
            history_display = history if history else "(根节点)"
            print(f"  {i+1}. {history_display}")
            print(f"       位置: {node.acting_position}, 行动数: {len(node.actions)}")

## 5. 验证总结

In [ ]:
total_nodes = sum(s.node_count() for s in strategies)
print(f"\n=== 验证总结 ===")
print(f"成功加载策略数: {len(strategies)}")
print(f"总节点数: {total_nodes}")
print(f"策略名称: {[s.name for s in strategies]}")